# Pars form hex log file for 16 Bit Mode

## 1. Probedaten einlesen 

In [9]:
from pathlib import Path

# Pfad festlegen
project_path = Path.cwd().parent
folder = 'example_output'
filename = 'hex_24102025_1434'
filepath = project_path / folder / filename

# Datei einlesen
try:
    with open(filepath, 'r') as f:
        file = f.read()
except Exception as e:
    print(f"Fehler beim Lesen der Datei: {e}")

# Checken
print(f"File output: {file[:100]}")
print(type(file))

File output: b'\x00d\x00z\x00x\x00m\x00d\x00d\x00q\x00q\x00m\x00p\x00u\x00z\x00\x80\x00\x85\x00\x88\x00\x80\x00p\
<class 'str'>


## 2. Sting in Bin umwandeln 

In [10]:
import ast

# Umwandlung von String zu Binärdaten
try:
    # bytes-Objekt erstellen 
    binary_data = ast.literal_eval(file)
    
    print(f"File output: {binary_data[:50]}")
    print(type(binary_data))
    
except (ValueError, SyntaxError) as e:
    print(f"Fehler beim Umwandeln des Strings in Binärdaten: {e}")

File output: b'\x00d\x00z\x00x\x00m\x00d\x00d\x00q\x00q\x00m\x00p\x00u\x00z\x00\x80\x00\x85\x00\x88\x00\x80\x00p\x00r\x00|\x00\x89\x00\x99\x00\xa2\x00\xa1\x00\x88\x00g'
<class 'bytes'>


## 3.1 Umwandlung von Char (16 bit Mode): 

In [ ]:
# Nur jedes Zweite Byte verwenden (für 16 bit Modus) 
bytes_list_of_char = binary_data[1::2]          # startet bei Index 1 und springt in 2er-Schritten
bytes_list_of_char = bytes_list_of_char[:10]        # ersten 8+2=10 Byte anschauen 
print(f"Verarbeitete Bytes: {bytes_list_of_char}")
print(type(bytes_list_of_char))
print(f"Test ob alle ungeraden 0er sind: {binary_data[0::2]}") # Nein sind es nicht 

# Header decoden 
str_of_char = bytes_list_of_char.decode('ascii')
print(f"\nString: {str_of_char}")
print(type(str_of_char))

# Header interpretieren 
magic  = str_of_char[0:8]
packet_id  = str_of_char[8:10]
print(f"\n{magic=} Länge: {len(magic)}")
print(f"{packet_id=} Länge: {len(packet_id)}")


Verarbeitete Bytes: b'dzxmddqqmp'
<class 'bytes'>
Test = b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x01\x02\x01\x01\x01\x01\x01\x01\x01\x01\x02\x02\x02\x02\x02\x02\x02\x02\x02\x02\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x01\x00\x00\x00\x00\x00\x01\x01\x01\x00\

# 3.2 Umwandeln von uint32 (16 Bit Mode):

In [12]:
import struct

# --- Jetzt kommt das Length-Feld (uint32) ---
# Jedes "echte" Byte ist 2 Bytes im 16-Bit-Datenstrom.
# Nach den 10 ASCII-Zeichen (10 × 2 = 20 Byte) kommt das Length-Feld (4 Byte = 8 Byte Rohdaten).

length_start = 20                  # Startposition im 16-Bit-Datenstrom
length_raw = binary_data[length_start:length_start + 8]   # 8 Byte im 16-Bit-Format (weil jedes Byte = 2 Byte)
length_low = length_raw[1::2]      # Nur die Low-Bytes extrahieren (echte Nutzdaten)

# Length als Little-Endian uint32 interpretieren
length_value = struct.unpack('<I', length_low)[0]

# Ausgabe
print("\n--- Length-Feld (uint32) ---")
print(f"Rohdaten (16-Bit): {length_raw}")
print(f"Nutzbytes (Low-Bytes): {length_low}")
print(f"Length (uint32, dezimal): {length_value}")
print(f"Length (uint32, hex): 0x{length_value:08X}")
print(f"Length (uint32, binär): {' '.join(f'{b:08b}' for b in length_low)}")


--- Length-Feld (uint32) ---
Rohdaten (16-Bit): b'\x00u\x00z\x00\x80\x00\x85'
Nutzbytes (Low-Bytes): b'uz\x80\x85'
Length (uint32, dezimal): 2239789685
Length (uint32, hex): 0x85807A75
Length (uint32, binär): 01110101 01111010 10000000 10000101


## Problem 

Warum haben wir im bin noch u und Z hinten ? Das sind keine hex Zahlen 
\x00d\x00z\x00x\x00m\x00d\x00d\x00q\x00q  \x00m\x00p  \x00u\x00z